In [38]:
import numpy as np
import pandas as pd
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
import lightgbm as lgb
import optuna

/Users/andyzhu/PycharmProjects/Kaggle_Spaceship_Titanic/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [66]:
train = pd.read_csv("Train.csv", index_col="PassengerId")
test = pd.read_csv("Test.csv", index_col="PassengerId")

In [3]:
train.columns

Index(['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'Transported',
       'GroupSize', 'Deck', 'Num', 'Side', 'Expenses'],
      dtype='str')

In [4]:
train.isna().sum().sort_values(ascending=False)

CryoSleep      217
VIP            203
HomePlanet     201
Deck           199
Num            199
Side           199
Destination    182
Age            179
Transported      0
GroupSize        0
Expenses         0
dtype: int64

In [67]:
train["CryoSleep"] = (train["CryoSleep"] == "True").astype(int)
test["CryoSleep"] = (test["CryoSleep"] == "True").astype(int)

train["VIP"] = (train["VIP"] == "True").astype(int)
test["VIP"] = (test["VIP"] == "True").astype(int)

train["Side"] = (train["Side"] == "S").astype(int)
test["Side"] = (test["Side"] == "S").astype(int)

In [ ]:
#Jetzt Null Werte bearbeiten

In [68]:
categorical_cols = ["Deck", "HomePlanet", "Side", "Destination"]

In [17]:
categorical_cols

['Deck', 'HomePlanet', 'Side', 'Destination']

In [13]:
#Cryosleep imputer
class GroupedModeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Modus pro Deck-Gruppe merken (nur aus Trainingsdaten!)
        self.group_modes_ = (
            X.groupby(self.group_col)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
        )
        # Fallback für Decks, die im Train nicht vorkamen
        self.global_mode_ = X[self.target_col].mode().iloc[0]
        return self

    def transform(self, X):
        X = X.copy()
        missing = X[self.target_col].isna()
        fill_values = X.loc[missing, self.group_col].map(self.group_modes_)
        fill_values = fill_values.fillna(self.global_mode_)  # falls Deck unbekannt oder Deck-Modus selbst NaN
        X.loc[missing, self.target_col] = fill_values
        return X

In [23]:
cat_cols = ["Deck", "HomePlanet", "Side", "Destination"]

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

ct = ColumnTransformer([
    ("cat", cat_pipe, cat_cols),
    ("age_impute", SimpleImputer(strategy="median"), ["Age", "Num"]),
], remainder="passthrough")

In [24]:
X = train.drop(columns=["Transported"])
y = train["Transported"]

In [35]:
Baseline = Pipeline([
    ("cryo_impute", GroupedModeImputer(group_col="Deck", target_col="CryoSleep")),
    ("preprocess", ct),
    ("scaler", StandardScaler()),
    ("model", LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 20),
        cv=5,
        scoring="accuracy",
        max_iter=1000,
    )),
])

In [36]:
cross_val_score(Baseline, X, y, cv=5, scoring="accuracy").mean()

/Users/andyzhu/PycharmProjects/Kaggle_Spaceship_Titanic/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/andyzhu/PycharmProjects/Kaggle_Spaceship_Titanic/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegress

np.float64(0.6853778906835735)

In [62]:
X_train_full = train.drop(columns=["Transported"])
y_train_full = train["Transported"]
X_train_full["HomePlanet"] = X_train_full["HomePlanet"].astype("category")
X_train_full["Destination"] = X_train_full["Destination"].astype("category")
X_train_full["Deck"] = X_train_full["Deck"].astype("category")

test["HomePlanet"] = test["HomePlanet"].astype("category")
test["Destination"] = test["Destination"].astype("category")
test["Deck"] = test["Deck"].astype("category")

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full
)

In [56]:
X_train_full.info()

<class 'pandas.DataFrame'>
Index: 8693 entries, 0001_01 to 9280_02
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   HomePlanet   8492 non-null   category
 1   CryoSleep    8693 non-null   int64   
 2   Destination  8511 non-null   category
 3   Age          8514 non-null   float64 
 4   VIP          8693 non-null   int64   
 5   GroupSize    8693 non-null   int64   
 6   Deck         8494 non-null   category
 7   Num          8494 non-null   float64 
 8   Side         8693 non-null   int64   
 9   Expenses     8693 non-null   float64 
dtypes: category(3), float64(3), int64(4)
memory usage: 826.9+ KB


In [57]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1,
    }

    pipe = Pipeline([
        ("cryo_impute", GroupedModeImputer(group_col="Deck", target_col="CryoSleep")),
        ("model", lgb.LGBMClassifier(**params)),
    ])

    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    return scores.mean()

In [59]:
print(X_train.dtypes)
print(X_train.iloc[[0,1,2]].dtypes)

HomePlanet     category
CryoSleep         int64
Destination    category
Age             float64
VIP               int64
GroupSize         int64
Deck           category
Num             float64
Side              int64
Expenses        float64
dtype: object
HomePlanet     category
CryoSleep         int64
Destination    category
Age             float64
VIP               int64
GroupSize         int64
Deck           category
Num             float64
Side              int64
Expenses        float64
dtype: object


In [60]:
study = optuna.create_study(
    direction="maximize",
    storage="sqlite:///spaceship_lgbm.db",
    study_name="spaceship_lgbm_v1",
    load_if_exists=True,  # falls du die Study später fortsetzen willst
)
study.optimize(objective, n_trials=1)
print(study.best_params)
print(study.best_value)

[I 2026-06-29 10:48:17,753] Using an existing study with name 'spaceship_lgbm_v1' instead of creating a new one.
[I 2026-06-29 10:48:23,136] Trial 3 finished with value: 0.7608629568585703 and parameters: {'n_estimators': 309, 'learning_rate': 0.012650412988243421, 'num_leaves': 246, 'max_depth': 7, 'min_child_samples': 20, 'subsample': 0.6789349085591438, 'colsample_bytree': 0.8470333729488162, 'reg_alpha': 0.0001182169602693255, 'reg_lambda': 3.0255493048005104}. Best is trial 3 with value: 0.7608629568585703.
[I 2026-06-29 10:48:23,990] Trial 4 finished with value: 0.7555855549641183 and parameters: {'n_estimators': 195, 'learning_rate': 0.04423687430621081, 'num_leaves': 45, 'max_depth': 4, 'min_child_samples': 96, 'subsample': 0.7426662040611244, 'colsample_bytree': 0.5257791240500085, 'reg_alpha': 3.133541764229452, 'reg_lambda': 4.919375431118348e-07}. Best is trial 3 with value: 0.7608629568585703.
[I 2026-06-29 10:48:24,743] Trial 5 finished with value: 0.75802072921467 and pa

{'n_estimators': 135, 'learning_rate': 0.021489533625375843, 'num_leaves': 31, 'max_depth': 11, 'min_child_samples': 9, 'subsample': 0.5544733523858567, 'colsample_bytree': 0.9893214678129948, 'reg_alpha': 1.4477184633018966e-06, 'reg_lambda': 0.05236252687598156}
0.7635691335708652


In [71]:
final_imputer = GroupedModeImputer(group_col="Deck", target_col="CryoSleep")
X_train_final = final_imputer.fit_transform(X_train_full)
X_test_final = final_imputer.transform(test)
best_params = study.best_params

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train_final, y_train_full)

,max_depth,11
,learning_rate,0.021489533625375843
,n_estimators,135
,min_child_samples,9
,subsample,0.5544733523858567
,colsample_bytree,0.9893214678129948
,reg_alpha,1.4477184633018966e-06
,reg_lambda,0.05236252687598156
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000


In [72]:
for col in ["HomePlanet", "Destination", "Deck"]:
    all_cats = pd.api.types.union_categoricals(
        [X_train_final[col].astype("category"), X_test_final[col].astype("category")]
    ).categories

    X_train_final[col] = X_train_final[col].astype(pd.CategoricalDtype(categories=all_cats))
    X_test_final[col] = X_test_final[col].astype(pd.CategoricalDtype(categories=all_cats))

In [73]:
# Prediction
preds = final_model.predict(X_test_final)

In [74]:
submission = pd.DataFrame({
    "PassengerId": X_test_final.index,
    "Transported": preds.astype(bool),
})

submission.to_csv("submission_lgb.csv", index=False)